# Exact CVRP with Pyomo — walkthrough

This notebook solves the Capacitated Vehicle Routing Problem **exactly**, as a
MILP, using the two-index vehicle-flow formulation with Miller-Tucker-Zemlin
subtour elimination (Toth & Vigo, *The Vehicle Routing Problem*, SIAM 2002 —
a standard textbook formulation, not one invented here).

Exact means no heuristics and no approximation: when the solver says optimal,
no shorter set of routes exists. It also means this does not scale, because
CVRP is NP-hard. The last section measures exactly how badly.

**The sample instances are synthetic** — random points on a square, not TSPLIB
or any other standard benchmark set. Objective values here are not comparable
to published results.

In [ ]:
import json
from pathlib import Path

from cvrp_opt.data.loaders import load_instance_json
from cvrp_opt.data.schema import Fleet, System
from cvrp_opt.model.builder import build_from_system
from cvrp_opt.solve import solve_cvrp
from cvrp_opt.viz import plot_benchmark, plot_capacity_utilization, plot_routes

REPO = Path.cwd().parent
system = load_instance_json(REPO / "data" / "sample_instances" / "cvrp_10.json")

print(f"{system.n_customers} customers, total demand {system.total_demand:g}")
print(f"vehicle capacity {system.fleet.capacity:g}")
print(f"k_min = ceil({system.total_demand:g} / {system.fleet.capacity:g}) = {system.k_min}")
print(f"fleet size in force: K = {system.num_vehicles}")

## Solving

`build_from_system` constructs the Pyomo model; `solve_cvrp` runs HiGHS and
turns the selected arcs back into ordered per-vehicle routes.

The MILP returns a set of arcs, not routes — reconstructing the tours is a
separate post-processing step, and one that is easy to get quietly wrong, so it
refuses to guess when the arcs are not a clean set of depot-to-depot paths.

In [ ]:
result = solve_cvrp(build_from_system(system))

print(result.summary())
print()
for i, route in enumerate(result.routes):
    stops = " -> ".join(str(c) for c in route)
    print(
        f"Vehicle {i + 1}: depot -> {stops} -> depot\n"
        f"    load {result.route_loads[i]:g}/{system.fleet.capacity:g}, "
        f"distance {result.route_distances[i]:.1f}"
    )

In [ ]:
plot_routes(system, result)

In [ ]:
plot_capacity_utilization(system, result)

## Fleet size: why `<= K` rather than `= K`

The brief's section 1.5 writes the depot degree constraint as *exactly* `K`
vehicles leaving. This model uses `<= K` instead (see `PROJECT_BRIEF.md`
section 7.1).

The difference only shows up when more vehicles are offered than the minimum.
Under equality every surplus vehicle is forced to make a trip, so handing the
planner a bigger fleet makes the answer *worse*. Under inequality the extras
stay parked.

In [ ]:
print(f"{'K offered':>10}  {'vehicles used':>13}  {'total distance':>14}")
for k in range(system.k_min, system.k_min + 4):
    variant = System(
        depot=system.depot,
        customers=system.customers,
        fleet=Fleet(capacity=system.fleet.capacity, num_vehicles=k),
    )
    variant_result = solve_cvrp(build_from_system(variant))
    print(
        f"{k:>10}  {variant_result.vehicles_used:>13}  "
        f"{variant_result.total_distance:>14.2f}"
    )

## What exactness costs

CVRP generalizes the Travelling Salesman Problem, so it is NP-hard. Solving it
exactly means branch-and-bound, and branch-and-bound on an NP-hard problem is
worst-case exponential in the instance size.

MTZ compounds this. It is compact and readable — one constraint family doing
both subtour elimination and capacity — but its linear relaxation is weak, so
the lower bound rises slowly and the search tree stays large.

The table below is **measured, not estimated**: it comes from
`docs/benchmark_results.json`, written by an actual run of
`python -m cvrp_opt.benchmark`. Rows that hit the time limit report a lower
bound on solve time, not a solve time.

In [ ]:
payload = json.loads((REPO / "docs" / "benchmark_results.json").read_text())
rows = payload["results"]

env = payload["environment"]
print(f"measured {env['generated_utc']} on {env['platform']}")
print(f"solver {env['solver']} (highspy {env['highspy_version']}), "
      f"limit {env['time_limit_seconds']:.0f}s per instance\n")

print(f"{'n':>4}  {'distance':>10}  {'bound':>10}  {'gap':>7}  {'time':>9}  proven")
for row in rows:
    print(
        f"{row['n_customers']:>4}  {row['total_distance']:>10.1f}  "
        f"{row['lower_bound']:>10.1f}  {row['gap']:>6.1%}  "
        f"{row['solve_time']:>8.1f}s  {'yes' if row['proven'] else 'NO (time limit)'}"
    )

In [ ]:
plot_benchmark(rows)

## Scope and caveats

- **Exact only.** No heuristics or metaheuristics. That is a deliberate scope
  decision, not an omission — a heuristic comparison would belong in a separate
  repo.
- **MTZ only.** Lazy-constraint or branch-and-cut subtour elimination is
  stronger, but needs solver callbacks that free solvers do not expose.
- **Single depot, homogeneous fleet.** No time windows, no multiple depots, no
  pickup-and-delivery, no split deliveries.
- **Euclidean distances kept as floats**, not rounded to integers the way
  TSPLIB's EUC_2D convention does — so objective values are not directly
  comparable to that literature.

Full formulation and citations in `docs/formulation.md`.